<a href="https://colab.research.google.com/github/anndhc/pengolahan_citra/blob/main/Jobsheet_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Segmentasi Menggunakan Thresholding Global dan Otsu

import matplotlib.pyplot as plt
from skimage import data, filters
from skimage.color import rgb2gray

# Memuat citra (contoh: coins)
image_coins = data.coins()  # Citra sudah grayscale

# Thresholding Global (manual)
thresh_manual = 100
binary_manual = image_coins > thresh_manual

# Thresholding Otsu
thresh_otsu = filters.threshold_otsu(image_coins)
binary_otsu = image_coins > thresh_otsu

# Visualisasi Hasil
fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_coins, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(binary_manual, cmap=plt.cm.gray)
ax[1].set_title(f'Manual Threshold (T={thresh_manual})')
ax[1].axis('off')

ax[2].imshow(binary_otsu, cmap=plt.cm.gray)
ax[2].set_title(f'Otsu\'s Threshold (T={thresh_otsu:.2f})')
ax[2].axis('off')

plt.tight_layout()
plt.show()

# Menampilkan nilai threshold Otsu
print(f"Nilai threshold Otsu yang ditemukan: {thresh_otsu}")


In [ ]:
# 2. Segmentasi Menggunakan Region Growing (Contoh Sederhana)

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, segmentation

# Memuat citra (contoh: camera)
image_camera = data.camera()

# Tentukan titik 'seed' (benih)
seed_point = (50, 150)  # Titik awal di area langit

# Terapkan algoritma flood fill
flood_mask = segmentation.flood(image_camera, seed_point, tolerance=10)

# Buat citra tersegmentasi (tandai region yang 'tumbuh')
segmented_image = np.copy(image_camera)
segmented_image[flood_mask] = 255  # Tandai region dengan warna putih

# Visualisasi Hasil
fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_camera, cmap=plt.cm.gray)
ax[0].plot(seed_point[1], seed_point[0], 'ro')  # Titik seed ditandai merah
ax[0].set_title('Original Image with Seed')
ax[0].axis('off')

ax[1].imshow(flood_mask, cmap=plt.cm.gray)
ax[1].set_title('Flood Fill Mask (Region)')
ax[1].axis('off')

ax[2].imshow(segmented_image, cmap=plt.cm.gray)
ax[2].set_title('Segmented Image (Region Marked)')
ax[2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# 2. Segmentasi Menggunakan Region Growing (Contoh Sederhana) dengan mengubah seed dan tolerance
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, segmentation

# Memuat citra
image_camera = data.camera()

# Konfigurasi percobaan
seed_points = [(50, 150), (200, 200)]           # Titik di langit dan di baju
tolerances = [10, 50]                           # Tolerance kecil dan besar

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
axes = axes.reshape(2, 3)

for i, (seed_point, tol) in enumerate(zip(seed_points, tolerances)):
    flood_mask = segmentation.flood(image_camera, seed_point, tolerance=tol)
    segmented_image = np.copy(image_camera)
    segmented_image[flood_mask] = 255

    axes[i][0].imshow(image_camera, cmap=plt.cm.gray)
    axes[i][0].plot(seed_point[1], seed_point[0], 'ro')
    axes[i][0].set_title(f'Original (Seed: {seed_point})')
    axes[i][0].axis('off')

    axes[i][1].imshow(flood_mask, cmap=plt.cm.gray)
    axes[i][1].set_title(f'Mask (Tolerance: {tol})')
    axes[i][1].axis('off')

    axes[i][2].imshow(segmented_image, cmap=plt.cm.gray)
    axes[i][2].set_title('Segmented Region')
    axes[i][2].axis('off')

plt.tight_layout()
plt.show()




In [ ]:
# 3. Segmentasi Citra Berwarna Menggunakan K-Means Clustering
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from sklearn.cluster import KMeans
from skimage.color import rgb2lab, lab2rgb
import warnings

# Memuat citra berwarna (contoh: astronaut)
image_astro = data.astronaut()
image_astro_float = image_astro.astype(float) / 255.0

# Konversi ke ruang warna Lab
image_lab = rgb2lab(image_astro_float)
rows, cols, dims = image_lab.shape
pixel_features = image_lab.reshape(rows * cols, dims)

# Terapkan K-Means dengan K=4
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pixel_labels = kmeans.fit_predict(pixel_features)

# Bentuk kembali label menjadi gambar
segmented_labels = pixel_labels.reshape(rows, cols)

# Bangun citra berdasarkan warna pusat masing-masing klaster
segmented_image_kmeans = np.zeros_like(image_lab)
centers_lab = kmeans.cluster_centers_
for k in range(n_clusters):
    mask_k = (pixel_labels == k).reshape(rows, cols)
    segmented_image_kmeans[mask_k] = centers_lab[k]

# Konversi hasil ke RGB
segmented_image_rgb = lab2rgb(segmented_image_kmeans)

# Visualisasi hasil
fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_astro)
ax[0].set_title('Original Image (RGB)')
ax[0].axis('off')

ax[1].imshow(segmented_labels, cmap='viridis')
ax[1].set_title(f'K-Means Labels (K={n_clusters})')
ax[1].axis('off')

ax[2].imshow(segmented_image_rgb)
ax[2].set_title(f'Segmented Image (K-Means, K={n_clusters})')
ax[2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# 4. Segmentasi Berbasis Tepi Menggunakan Watershed
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters, segmentation, morphology, measure
from scipy import ndimage as ndi

# 1. Memuat citra (contoh: coins)
image_coins = data.coins()

# 2. Hitung gradien citra (sebagai 'topografi')
elevation_map = filters.sobel(image_coins)

# 3. Tentukan marker (penanda awal untuk setiap cekungan/objek)
# Kita bisa menggunakan thresholding untuk mendapatkan marker kasar
markers = np.zeros_like(image_coins)
markers[image_coins < 30] = 1  # Marker untuk latar belakang
markers[image_coins > 150] = 2  # Marker untuk objek (koin)

# 4. Terapkan algoritma Watershed
segmentation_watershed = segmentation.watershed(elevation_map, markers)

# Warnai hasil segmentasi untuk visualisasi
segmented_colored = segmentation.mark_boundaries(image_coins, segmentation_watershed)

# 5. Visualisasi Hasil
fig, axes = plt.subplots(ncols=4, figsize=(16, 4), sharex=True, sharey=True)
ax = axes.ravel()

ax[0].imshow(image_coins, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(elevation_map, cmap=plt.cm.nipy_spectral)
ax[1].set_title('Elevation Map (Sobel Gradient)')
ax[1].axis('off')

ax[2].imshow(markers, cmap=plt.cm.nipy_spectral)
ax[2].set_title('Markers')
ax[2].axis('off')

ax[3].imshow(segmented_colored)
ax[3].set_title('Watershed Segmentation')
ax[3].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# 5. Perbandingan Visual Hasil Segmentasi
import matplotlib.pyplot as plt
from skimage import data, filters, segmentation, img_as_float, color
from sklearn.cluster import KMeans
import numpy as np
import warnings

# 1. Pilih satu citra untuk perbandingan (misal: camera)
image = data.camera()
image_float = img_as_float(image)

# 2. Lakukan beberapa metode segmentasi
# a) Otsu Thresholding
thresh_otsu = filters.threshold_otsu(image)
binary_otsu = image > thresh_otsu

# b) K-Means (misal K=3)
# Reshape untuk K-Means (1 fitur: intensitas)
rows, cols = image.shape
pixel_features = image_float.reshape(rows * cols, 1)
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pixel_labels = kmeans.fit_predict(pixel_features)
segmented_kmeans_labels = pixel_labels.reshape(rows, cols)

# c) Watershed (gunakan marker sederhana dari Otsu)
elevation_map = filters.sobel(image)
markers = np.zeros_like(image)
markers[image < thresh_otsu] = 1
markers[image > thresh_otsu] = 2
segmentation_watershed = segmentation.watershed(elevation_map, markers)

# 3. Visualisasi Perbandingan
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
ax = axes.ravel()

ax[0].imshow(image, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(binary_otsu, cmap=plt.cm.gray)
ax[1].set_title('Otsu Thresholding')
ax[1].axis('off')

ax[2].imshow(segmented_kmeans_labels, cmap='viridis') # Gunakan cmap berbeda untuk label
ax[2].set_title(f'K-Means (K={n_clusters})')
ax[2].axis('off')

# Gunakan mark_boundaries untuk Watershed agar lebih jelas
segmented_watershed_colored = segmentation.mark_boundaries(image_float, segmentation_watershed, color=(1,0,0)) # Batas merah
ax[3].imshow(segmented_watershed_colored)
ax[3].set_title('Watershed')
ax[3].axis('off')

plt.suptitle('Comparison of Segmentation Methods')
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
plt.show()


In [ ]:
# Eksperimen Eksperimen Thresholding pada data.page()
import matplotlib.pyplot as plt
from skimage import data, filters
import numpy as np

# Ambil citra dokumen (page)
image = data.page()

# Otsu Thresholding
thresh_otsu = filters.threshold_otsu(image)
binary_otsu = image > thresh_otsu

# Threshold Local
block_size = 35
binary_local = image > filters.threshold_local(image, block_size)

# Threshold Yen
thresh_yen = filters.threshold_yen(image)
binary_yen = image > thresh_yen

# Visualisasi
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
titles = ['Original', 'Otsu', 'Local', 'Yen']
images = [image, binary_otsu, binary_local, binary_yen]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from skimage import io, color, filters, segmentation
from sklearn.cluster import KMeans
import numpy as np

# 1. Baca citra apel dan pastikan hanya RGB
image_rgb = io.imread('apel.jpg')
if image_rgb.shape[2] == 4:
    image_rgb = image_rgb[:, :, :3]

# 2. Konversi ke grayscale
image_gray = color.rgb2gray(image_rgb)

# --- Metode 1: Otsu Thresholding ---
thresh = filters.threshold_otsu(image_gray)
binary_otsu = image_gray > thresh

# --- Metode 2: K-Means Clustering (K=3) ---
rows, cols, _ = image_rgb.shape
flat_img = image_rgb.reshape((-1, 3))
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10).fit(flat_img)
seg_kmeans = kmeans.labels_.reshape(rows, cols)

# --- Metode 3: Watershed Segmentation ---
elevation_map = filters.sobel(image_gray)
markers = np.zeros_like(image_gray, dtype=np.int32)
markers[image_gray < 0.4] = 1
markers[image_gray > 0.8] = 2
ws = segmentation.watershed(elevation_map, markers)
ws_boundaries = segmentation.mark_boundaries(image_rgb, ws)

# --- Visualisasi Hasil ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ['Original', 'Otsu Thresholding', 'K-Means Clustering', 'Watershed']
images = [image_rgb, binary_otsu, seg_kmeans, ws_boundaries]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray' if title != 'Original' else None)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()
